In [ ]:
import polars as pl




#--------------------------------------------------------------------------------------------------------------------
#General Configs :
#--------------------------------------------------------------------------------------------------------------------

pl.Config.set_tbl_rows(-1) #show all rows
pl.Config.set_tbl_cols(-1) #show all cols
pl.Config.set_fmt_str_lengths(50) #change format string length
pl.Config.set_tbl_width_chars(400) #change table width based on characters

#--------------------------------------------------------------------------------------------------------------------
#Reading the Input file to a lazyframe:
#--------------------------------------------------------------------------------------------------------------------
input_file = "2019-Nov.parquet"
lf1 = pl.scan_parquet(input_file)





#--------------------------------------------------------------------------------------------------------------------
#Change datatypes, Filter user_session Nulls, Remove Duplicates:
#--------------------------------------------------------------------------------------------------------------------
data_types = (lf1
        .with_columns(
            pl.col("event_time").str.to_datetime(format="%Y-%m-%d %H:%M:%S %Z"), #Year-month-day Hour:Minute:Second timeZone
            pl.col(["product_id", "user_id"]).cast(dtype=pl.Int32), #Int 64 → Int32
            pl.col("price").cast(dtype=pl.Float32) #Float64 → Float32
            
        )
        
        .filter(
            pl.col("user_session").is_not_null(),                         #filter where user_session is not NULL
                                                         
        ) 
        
        .unique()  #filter only unique rows 
)

# --------------------------------------------------------------------------------------------------------------------
# Split Category:
# --------------------------------------------------------------------------------------------------------------------
# let's check how many levels of catogories exist first 
# print(data_types.select(
#     pl.col("category_code").str.split(".").list.len())
#     .unique().collect())
 


split_category = (data_types

        .with_columns(
            pl.col("category_code").str.split(by=".").list.get(index=0).alias("main_category"), #split by ".", turn to a list a take first position (index = 0)
            pl.col("category_code").str.split(by=".").list.slice(offset=1).list.join(".").alias("sub_category") #split by ".", turn to a list a take everything from 2nd position onward then join them
        )
        
        .drop("category_code")
        
        
        .select([
            "event_time", "event_type", "product_id",
            "category_id", "main_category", "sub_category",  #used select to sort columns where new category columns sit besides category_id
            "brand", "price", "user_id", "user_session"
        ])
    
)


#--------------------------------------------------------------------------------------------------------------------
#Check Missing Values:
#--------------------------------------------------------------------------------------------------------------------


# total_rows = data_types.select(pl.len()).collect(engine="streaming").item() #count rows

# data_types_df = data_types.collect(engine="streaming") #collected to dataframe, because "transpose" is not available in lazyframe

# missing_values_percentages = (data_types_df
                              
#         .null_count() #null count for each column
#         .transpose(include_header = True) #flip the data (rows becomes columns and vise versa)
#         .rename({
#             "column" : "column_name",
#             "column_0" : "null_count"})
        
#         #since we used transpose ,it results to two columns, one called "column" and one called "column_0"
#         #we renamed "column" with "column_name"
#         #and we renamed "column_0" with "null_count"
        
        
        
#         .with_columns( #calculated percentage of nulls
#             ( pl.col("null_count") / total_rows * 100).round(2).alias("null_percentage")
#         )
        
                                        
# )

# print(missing_values_percentages)


#--------------------------------------------------------------------------------------------------------------------
#Replace Missing Values:
#--------------------------------------------------------------------------------------------------------------------

nulls_replace = (split_category
                 
        .with_columns(
            pl.col("main_category").fill_null("unknown_category"), #replace nulls in "main_category" column
            pl.col("sub_category").fill_null("unknown_subcategory"), #replace nulls in "sub_category" column
            pl.col("brand").fill_null("unknown_brand")             #replace nulls in "brand" column 
        )
        

    
)



#--------------------------------------------------------------------------------------------------------------------
#Sinking:
#--------------------------------------------------------------------------------------------------------------------
#since our data is heavy, i'm going to use batch processing method, after i completed standardizing and cleaning the data,
# i'm going to write that into a new file, so i don't have to run the process again each time 


cleaned_data = nulls_replace.sink_csv(path="cleaned_2019-Nov.parquet", engine="streaming")







In [ ]:
import polars as pl




#--------------------------------------------------------------------------------------------------------------------
#General Configs :
#--------------------------------------------------------------------------------------------------------------------

pl.Config.set_tbl_rows(-1) #show all rows
pl.Config.set_tbl_cols(-1) #show all cols
pl.Config.set_fmt_str_lengths(50) #change format string length
pl.Config.set_tbl_width_chars(400) #change table width based on characters

pl.Config.set_fmt_float("full") # tells Polars: "don't use scientific notation, show the full number 
#instead of "2.05242416e8" it types "205,242,416"

#--------------------------------------------------------------------------------------------------------------------
#Reading the Input file to a lazyframe:
#--------------------------------------------------------------------------------------------------------------------
input_file = "cleaned_2019-Nov.parquet"
lf2 = pl.scan_parquet(input_file)



#--------------------------------------------------------------------------------------------------------------------
#Calculating conversion rate (DROP-OFF rates at each step):
#--------------------------------------------------------------------------------------------------------------------


conversion_rate = (lf2
          
        .group_by(
            pl.col("event_type")           
        )
        
        .agg(
            pl.col("user_id").n_unique().sum().alias("unique users")
        )
        
    
)   


#collect values in a dictionary, since we only have 2 columns:

events = dict(conversion_rate
        
        .collect(engine="streaming")  
        .iter_rows()  
    
)

views = events.get("view", 0) # "0" means return 0 if there's no value instead of raising an error
carts = events.get("cart", 0) 
purchases = events.get("purchase", 0) 


#calculate conversion rates

view_to_cart_rate = (carts / views) * 100
cart_to_purchase_rate = (purchases/ carts) * 100

view_to_purchase_rate = (purchases/ views) * 100

#printing conversion rates
print ("Conversion Rate Values:\n")
print (f"view → cart rate: {view_to_cart_rate:.2f}%")
print (f"cart → purchase rate: {cart_to_purchase_rate:.2f}%")
print (f"view → purchase rate: {view_to_purchase_rate:.2f}%\n")


#conversion rate table
conversion_rate = pl.DataFrame({
    "type of conversion" : ["view → cart", "cart → purchase", "view → purchase"],
    "conversion rate" : [f"{view_to_cart_rate:.2f}%", f"{cart_to_purchase_rate:.2f}%", f"{view_to_purchase_rate:.2f}%" ]
}
)



#--------------------------------------------------------------------------------------------------------------------
#calculating TOP 5 BEST/WORST main Categories in REVENUE ?:
#--------------------------------------------------------------------------------------------------------------------



top_categories = (lf2
              
        .with_columns(
            pl.col("price").cast(pl.Float64) #cast price to float64 temporarily so i can round the sum
        ) 
                        
        .filter(
            pl.col("event_type") == "purchase" #filter only purchases    
        )
        
        .group_by("main_category")
        .agg(
            pl.col("price").sum().round(2).alias("total revenue"), #count total revenue
            pl.len().alias("total orders") #by counting the length of how many each category appears, i'll count it as total orders
        ) 
        
        .sort(by = "total revenue",descending=True) #sort in descending order by revenue
        
         
)

TOP_5 = top_categories.head(5).collect(engine="streaming")
WORST_5 = top_categories.tail(5).collect(engine="streaming")



print("here's table that show TOP 5 BEST Categories, based on revenue:")
print(TOP_5)

print("\n\nhere's table that show WORST 5 BEST Categories, based on revenue:")
print(WORST_5)


#--------------------------------------------------------------------------------------------------------------------
#calculating TOP 5 BEST/WORST main Categories in REVENUE ?:
#--------------------------------------------------------------------------------------------------------------------


LTV = (lf2
       
        .with_columns(
            pl.col("price").cast(pl.Float64) #cast price to float64 temporarily so i can round the sum
        ) 
        
        .filter(
            pl.col("event_type") == "purchase"
        )
        
        .group_by("user_id")
        .agg(
            pl.col("price").sum().alias("total_user_revenue")
        )
       
    
)


avg_LTV = (LTV

        .select(
            pl.col("total_user_revenue").mean().round(2).alias("average_LTV")
        )
        
        .collect(engine="streaming")
        .item()
        
        
        
)

print (f"\n\nAverage Customer LTV is: {avg_LTV}")








Conversion Rate Values:

view → cart rate: 22.36%
cart → purchase rate: 53.45%
view → purchase rate: 11.95%

here's table that show TOP 5 BEST Categories, based on revenue:
shape: (5, 3)
┌──────────────────┬───────────────┬──────────────┐
│ main_category    ┆ total revenue ┆ total orders │
│ ---              ┆ ---           ┆ ---          │
│ str              ┆ f64           ┆ u32          │
╞══════════════════╪═══════════════╪══════════════╡
│ electronics      ┆ 205249741.53  ┆ 493637       │
│ unknown_category ┆ 29879190.27   ┆ 234214       │
│ appliances       ┆ 18640385.14   ┆ 99023        │
│ computers        ┆ 13994330.68   ┆ 34477        │
│ furniture        ┆ 2543251.02    ┆ 11542        │
└──────────────────┴───────────────┴──────────────┘


here's table that show WORST 5 BEST Categories, based on revenue:
shape: (5, 3)
┌───────────────┬───────────────┬──────────────┐
│ main_category ┆ total revenue ┆ total orders │
│ ---           ┆ ---           ┆ ---          │
│ str       

### SHOWING TABLES IN BETTER WAY USING "great_tables" LIB

In [ ]:
from great_tables import GT
from IPython.display import HTML

#--------------------------------------------------------------------------------------------------------------------
#general table inputs
#--------------------------------------------------------------------------------------------------------------------

header_colors = ["#f57c00", "#1a3fa0", "#b71c1c", "#1e88e5"]   # colors repeat if there are more columns
stub_gradient = "linear-gradient(90deg, #0b3c91, #1e90ff)"     # dark blue to light blue first column (dates)
comma_columns = []                                       # only these columns get 1,234.50 style

show_first_header = 1      # 1 = show first column name, 0 = leave it empty

#--------------------------------------------------------------------------------------------------------------------
# reusable functions for showing tables, extracting it to html code:
#--------------------------------------------------------------------------------------------------------------------

def design_table(df, header_colors, stub_gradient, comma_columns=[], show_first_header=0):
    columns = df.columns    # list of column names
    rows = df.rows()        # list of tuples, one per row

    css = f"""
    <style>
    .dt {{ border-collapse: separate; border-spacing: 5px;
           font-family: Arial, sans-serif; font-size: 14px; }}
    .dt th {{ color: white; padding: 8px 24px; text-align: center;
              border-radius: 12px 12px 0 0; white-space: nowrap; }}
    .dt td {{ background: #e6e6e6; color: #444; padding: 8px 24px;
              text-align: center; white-space: nowrap; }}
    .dt tr:nth-child(even) td {{ background: #f2f2f2; }}
    .dt td.stub {{ background: {stub_gradient} !important; color: white;
                   font-weight: bold; text-align: left;
                   border-radius: 20px 0 0 20px; }}
    </style>
    """

    html = "<div style='overflow-x: auto;'>"        # scrolls sideways if the table is too wide
    html += css + '<table class="dt">'

    # Header row
    html += "<tr>"

    # First header cell: shown or empty, depending on the switch
    if show_first_header == 1:
        html += f"<th style='background: {stub_gradient}'>{columns[0]}</th>"
    else:
        html += "<th style='background: transparent'></th>"

    # Other header cells: one color each, colors repeat
    for index, column_name in enumerate(columns[1:]):
        color = header_colors[index % len(header_colors)]   # wrap around the color list
        html += f"<th style='background: {color}'>{column_name}</th>"
    html += "</tr>"

    # Body rows
    for row in rows:
        html += "<tr>"
        html += f"<td class='stub'>{row[0]}</td>"           # first value = row label

        for position, value in enumerate(row[1:]):
            column_name = columns[position + 1]             # +1 because we skipped the first column

            if value is None:                               # show empty instead of "None"
                text = ""
            elif column_name in comma_columns:              # format only the chosen columns
                text = f"{value:,.2f}"                      # 1234.5 -> 1,234.50
            else:
                text = str(value)                           # IDs stay exactly as they are

            html += f"<td>{text}</td>"
        html += "</tr>"

    html += "</table></div>"
    return HTML(html)


def save_table_html(table, filename, title="Table Preview"):
    page = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>{title}</title>
    <style>
        body {{ background: white; margin: 30px; }}
    </style>
</head>
<body>
    {table.data}
</body>
</html>"""

    with open(filename, "w", encoding="utf-8") as file:
        file.write(page)

    print(f"Saved to {filename}")




#--------------------------------------------------------------------------------------------------------------------
#preview of data after sinking (after cleaning and standardizing)
#--------------------------------------------------------------------------------------------------------------------

df_preview = top_categories.tail(5).collect()






table = design_table(df_preview, header_colors, stub_gradient, show_first_header=0)
display(table)

save_table_html(table, "WORST5_categories.html", "WORST Categories")


average_LTV
623.12


Saved to average_LTV_preview.html
